# Day 05 下午学生项目：电商用户多维分析

**小组编号：** 请填写  
**成员：** 请填写  
**专题方向：** A / B / C / D / E

> 请只在标有 `TODO` 的区域填写代码，不要删除任务说明、检查点和反思题。

## 实验目标与提交要求

你需要完成：

1. 数据加载与验收；
2. 公共基础指标；
3. 一个单维专题分析；
4. 一个双维交叉分析；
5. 三个CSV报表；
6. 至少3条结论、1条限制和1项建议。

**重要边界：** 一行是一名用户；返现不是消费金额；相关不等于因果。

## 任务0：小组配置

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

GROUP_ID = "请填写组号"
MEMBERS = ["请填写成员"]
TOPIC = "请填写A/B/C/D/E"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到清洗后数据，请检查项目目录。")

ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_student" / GROUP_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("小组：", GROUP_ID, MEMBERS)
print("专题：", TOPIC)
print("输入：", DATA_PATH)
print("输出：", OUTPUT_DIR)

小组： 请填写组号 ['请填写成员']
专题： 请填写A/B/C/D/E
输入： e:\pythonProject\output\day04_project\ecommerce_customer_cleaned.csv
输出： e:\pythonProject\output\day05_student\请填写组号


### 检查点0

- [ ] 已填写组号、成员和专题；
- [ ] Notebook文件名包含组号；
- [ ] 输出目录中的组号正确。

## 任务1：加载并验收数据（必做）

In [4]:
# TODO 1：读取清洗后的CSV，变量名必须为df
df = pd.read_csv(DATA_PATH)

# TODO 2：输出shape、前5行和字段类型
print("shape:",df.shape)
print("前五行数据")
print(df.head(5))
print("字段类型")
print(df.dtypes)
# TODO 3：计算以下验收结果
core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]
validation = {
    "行数": df.shape[0],
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}
validation

shape: (5630, 22)
前五行数据
   CustomerID  Churn  Tenure PreferredLoginDevice  CityTier  WarehouseToHome  \
0       50001      1    4.00         Mobile Phone         3             6.00   
1       50002      1    9.00         Mobile Phone         1             8.00   
2       50003      1    9.00         Mobile Phone         1            30.00   
3       50004      1    0.00         Mobile Phone         3            15.00   
4       50005      1    0.00         Mobile Phone         1            12.00   

  PreferredPaymentMode  Gender  HourSpendOnApp  NumberOfDeviceRegistered  \
0           Debit Card  Female            3.00                         3   
1                  UPI    Male            3.00                         4   
2           Debit Card    Male            2.00                         4   
3           Debit Card    Male            2.00                         4   
4          Credit Card    Male            3.00                         3   

     PreferedOrderCat  SatisfactionSco

{'行数': 5630, '列数': 22, 'CustomerID重复数': 0, '核心字段缺失数': 0, 'Churn取值': [0, 1]}

In [5]:
# 完成上一个单元后再运行本检查点
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应唯一"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("检查点1通过")

检查点1通过


**数据粒度：** 请用一句话填写：  
____________________________________________________________

## 任务2：公共基础指标（必做）

In [6]:
# TODO：构建overall_metrics DataFrame，至少包含以下指标：
# 用户数、流失人数、流失率、平均订单数、订单数中位数、
# 平均优惠券数、平均返现、平均App时长、平均满意度、平均距上次下单天数
overall_metrics = pd.DataFrame({
    "指标": ["用户数", "流失人数", "流失率", "平均订单数", "订单数中位数",
             "平均优惠券数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数"],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})
display(overall_metrics)

# TODO：将下面变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()

# display(overall_metrics)

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券数,1.72
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [7]:
# 检查点2
assert isinstance(overall_metrics, pd.DataFrame), "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, "公共指标至少10项"

# TODO：将下面变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, "总体流失率不正确"
print("检查点2通过")

检查点2通过


## 任务3：单维专题分析（必做）

请选择一个专题：

- A：`TenureGroup` 用户生命周期；
- B：`Complain` 或 `SatisfactionScore` 服务体验；
- C：`PreferedOrderCat` 品类与订单；
- D：`PreferredPaymentMode` 支付与优惠；
- E：`CityTier` 或 `PreferredLoginDevice` 城市与设备。

最低要求：使用 `groupby + agg`，同时输出用户数和至少3项业务指标。

In [8]:
# TODO：填写你的分组字段
segment_field = "TenureGroup"

# TODO：使用groupby + agg完成命名聚合
segment_analysis = (
    df.groupby(segment_field)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均返现=("CashbackAmount", "mean"),
      )
      .reset_index()
)

# TODO：重置索引、排序并展示
# display(segment_analysis)
segment_analysis = segment_analysis.sort_values("流失率", ascending=False)
display(segment_analysis)

,TenureGroup,用户数,流失率,平均订单数,平均返现
0,0-6月,1967,0.35,2.47,158.79
4,6-12月,1585,0.10,2.66,161.48
1,12-24月,1574,0.06,3.64,200.72
2,24-36月,500,0.00,3.70,225.29
3,36月以上,4,0.00,2.00,226.38


In [9]:
# 检查点3
assert segment_field in df.columns, "segment_field不是有效字段"
assert isinstance(segment_analysis, pd.DataFrame), "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, "专题表必须包含用户数"
assert len(segment_analysis) >= 2, "专题分析至少应有两个分组"
print("检查点3通过")

检查点3通过


### 专题分析记录

**数据现象：**  
请填写。

**可能解释：**  
请填写，并使用“可能、相关、需验证”等有边界的表达。

## 任务4：双维度交叉分析（必做）

In [10]:
# TODO：从以下维度中选择两个
# TenureGroup、Complain、PreferedOrderCat、CityTier、PreferredLoginDevice
dim_1 = "TenureGroup"
dim_2 = "Complain"

# TODO：按两个维度统计用户数、流失人数、流失率，以及至少一个行为指标
cross_analysis = (
    df.groupby([dim_1, dim_2], observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
      )
      .reset_index()
)

# TODO：新增"样本提示"列；用户数<30标记为"小样本"，否则为"可观察"
cross_analysis["样本提示"] = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察"
)
# TODO：按流失率或用户数排序并展示
cross_analysis = cross_analysis.sort_values("流失率", ascending=False)
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
1,0-6月,1,626,369,0.59,2.59,可观察
0,0-6月,0,1341,320,0.24,2.42,可观察
9,6-12月,1,393,83,0.21,2.66,可观察
3,12-24月,1,439,56,0.13,3.27,可观察
8,6-12月,0,1192,74,0.06,2.66,可观察
2,12-24月,0,1135,46,0.04,3.79,可观察
5,24-36月,1,144,0,0.00,3.40,可观察
4,24-36月,0,356,0,0.00,3.82,可观察
7,36月以上,1,2,0,0.00,1.50,小样本
6,36月以上,0,2,0,0.00,2.50,小样本


In [11]:
# 检查点4
assert dim_1 in df.columns and dim_2 in df.columns, "两个维度必须是有效字段"
assert dim_1 != dim_2, "两个维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), "cross_analysis应为DataFrame"
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns), "双维表缺少必需列"
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"}), "样本提示取值不正确"
print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的组合：**  
请填写。

**该组合的样本量与流失率：**  
请填写。

**为什么不能直接下因果结论：**  
请填写。

## 任务5：报表输出与回读验证（必做）

In [12]:
# TODO：将三个表导出到OUTPUT_DIR
# 文件名必须为：overall_metrics.csv、segment_analysis.csv、cross_analysis.csv
# 要求：index=False，encoding="utf-8-sig"

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

# TODO：循环导出并重新读取；打印每个文件的shape
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    check = pd.read_csv(path)
    print(f"已输出 {filename}: {check.shape}")

已输出 overall_metrics.csv: (10, 2)
已输出 segment_analysis.csv: (5, 5)
已输出 cross_analysis.csv: (10, 7)


In [13]:
# 检查点5
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    assert path.exists(), f"缺少输出文件：{filename}"
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"
print("检查点5通过：三个CSV均已成功导出并回读。")

检查点5通过：三个CSV均已成功导出并回读。


## 任务6：结论、限制与建议（必做）

### 结论1

在 0-6月新用户 中， 流失率 为 35%，与 24-36月用户的0% 相比 高出35个百分点。
对应证据表：segment_analysis。

### 结论2

在0-6月新用户中，有投诉的用户流失率高达58.95%，而无投诉用户仅为23.86%，
投诉与流失存在明显关联。对应证据表：cross_analysis。

### 结论3

不同生命周期的平均返现金额差异明显，12月以上用户平均返现超过200元，
而0-6月用户仅158.79元，可能与老用户消费更多有关，需结合消费金额进一步验证。
对应证据表：segment_analysis。

### 分析限制

缺少订单金额和订单日期，因此不能计算GMV、客单价或时间趋势；
返现金额不等于消费金额，不能直接衡量用户价值；
36月以上分组仅4人，样本量过小，相关结论不可靠。

### 运营建议与验证方式

建议：对0-6月新用户中的投诉用户进行主动干预（如优惠券补偿或专属客服），
降低早期流失率。

验证方式：需要设计A/B实验，将投诉新用户随机分为干预组和对照组，
对比两组30天后的留存率。同时需要收集投诉原因分类数据，定位具体问题。

## 拓展任务（选做）

In [ ]:
# 可选方向：
# 1. 使用qcut构建订单活跃度分层；
# 2. 设计供第6天绘图使用的长表；
# 3. 对反直觉结果提出两种数据核查方法。

# TODO（选做）

## 提交前检查

- [ ] 已填写组号、成员和专题；
- [ ] 已重启内核并从头运行成功；
- [ ] 所有比例表都包含样本量；
- [ ] 三个CSV已导出并回读；
- [ ] 至少3条结论可对应到具体表格；
- [ ] 已写明分析限制和验证建议；
- [ ] 没有把返现写成消费额，没有把相关写成因果。